In [0]:

# DAY 2 - BRONZE INGESTION
# Step 1: Verify raw landing files

bls_path = "/Volumes/workspace/rearc/rearc_landing/raw/bls/"
datausa_path = "/Volumes/workspace/rearc/rearc_landing/raw/datausa/"

print("BLS RAW FILES")
print("-" * 50)

bls_files = dbutils.fs.ls(bls_path)

for file in bls_files:
    print(file.name, file.size, "bytes")

print("\nDATAUSA RAW FILES")
print("-" * 50)

datausa_files = dbutils.fs.ls(datausa_path)

for file in datausa_files:
    print(file.name, file.size, "bytes")

assert len(bls_files) >= 12, "BLS files are missing"
assert any(f.name == "population.json" for f in datausa_files), \
    "DataUSA population.json is missing"

print("\nRAW LANDING VALIDATION PASSED")


In [0]:

import re

from pyspark.sql.functions import (
    current_timestamp,
    col
)

# 1. Read BLS historical data
bls_data_file = (
    "/Volumes/workspace/rearc/rearc_landing/raw/bls/"
    "pr.data.1.AllData"
)

bls_df = (
    spark.read
    .option("header", True)
    .option("sep", "\t")
    .option("inferSchema", False)
    .csv(bls_data_file)
)

# 2. Capture source file metadata
bronze_bls_df = bls_df.select(
    "*",
    col("_metadata.file_path").alias("_source_file")
)

# 3. Standardize column names
def clean_column_name(name):
    name = name.strip().lower()
    name = re.sub(r"[^a-zA-Z0-9_]", "_", name)
    name = re.sub(r"_+", "_", name)
    return name.strip("_")

cleaned_names = [
    clean_column_name(c)
    for c in bronze_bls_df.columns
]

# Ensure column names are unique
assert len(cleaned_names) == len(set(cleaned_names)), \
    "Duplicate column names after standardization"

bronze_bls_df = bronze_bls_df.toDF(*cleaned_names)

# 4. Add ingestion timestamp
bronze_bls_df = bronze_bls_df.withColumn(
    "_ingestion_timestamp",
    current_timestamp()
)

# 5. Write Bronze Delta table
(
    bronze_bls_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.rearc.bronze_bls_data")
)

# 6. Validate
print("BLS BRONZE TABLE CREATED")
print("Total records:", bronze_bls_df.count())
print("Columns:", bronze_bls_df.columns)

display(bronze_bls_df.limit(10))


In [0]:

from pyspark.sql.functions import (
    col,
    explode,
    current_timestamp
)

# DataUSA source file
datausa_file = (
    "/Volumes/workspace/rearc/rearc_landing/raw/datausa/"
    "population.json"
)

# Read multiline JSON
datausa_raw_df = (
    spark.read
    .option("multiLine", True)
    .json(datausa_file)
)

# Extract records from the JSON data array
datausa_exploded_df = datausa_raw_df.select(
    explode(col("data")).alias("record"),
    col("_metadata.file_path").alias("_source_file")
)

# Select and standardize columns
bronze_datausa_df = datausa_exploded_df.select(
    col("record").getField("Nation ID").alias("nation_id"),
    col("record").getField("Nation").alias("nation"),
    col("record").getField("Year").alias("year"),
    col("record").getField("Population").alias("population"),
    col("_source_file"),
    current_timestamp().alias("_ingestion_timestamp")
)

# Write to Delta
(
    bronze_datausa_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.rearc.bronze_datausa_population")
)

print("DATAUSA BRONZE TABLE CREATED")
print("Total records:", bronze_datausa_df.count())

display(bronze_datausa_df.orderBy("year"))


In [0]:

# Validate Bronze tables

bls_bronze = spark.table("workspace.rearc.bronze_bls_data")
datausa_bronze = spark.table(
    "workspace.rearc.bronze_datausa_population"
)

bls_count = bls_bronze.count()
datausa_count = datausa_bronze.count()

print("BRONZE LAYER VALIDATION")
print("-" * 40)

print("BLS records:", bls_count)
print("DataUSA records:", datausa_count)

# Check nulls in important columns
bls_nulls = bls_bronze.filter(
    col("series_id").isNull() |
    col("year").isNull() |
    col("period").isNull() |
    col("value").isNull()
).count()

datausa_nulls = datausa_bronze.filter(
    col("year").isNull() |
    col("population").isNull()
).count()

print("BLS critical null records:", bls_nulls)
print("DataUSA critical null records:", datausa_nulls)

assert bls_count == 77126
assert datausa_count == 11
assert bls_nulls == 0
assert datausa_nulls == 0

print("\nBRONZE VALIDATION PASSED")


In [0]:

# Inspect BLS metadata files before Silver transformations

metadata_files = [
    "pr.series",
    "pr.sector",
    "pr.class",
    "pr.measure",
    "pr.duration",
    "pr.seasonal",
    "pr.period"
]

metadata_path = (
    "/Volumes/workspace/rearc/rearc_landing/raw/bls/"
)

for file_name in metadata_files:
    print("\n" + "=" * 60)
    print("FILE:", file_name)
    print("=" * 60)

    df = (
        spark.read
        .option("header", True)
        .option("sep", "\t")
        .option("inferSchema", False)
        .csv(metadata_path + file_name)
    )

    print("Columns:", df.columns)
    display(df.limit(5))


In [0]:

import re
from pyspark.sql.functions import col, current_timestamp

# BLS reference files
metadata_files = {
    "pr.series": "bronze_bls_series",
    "pr.sector": "bronze_bls_sector",
    "pr.class": "bronze_bls_class",
    "pr.measure": "bronze_bls_measure",
    "pr.duration": "bronze_bls_duration",
    "pr.seasonal": "bronze_bls_seasonal",
    "pr.period": "bronze_bls_period"
}

base_path = (
    "/Volumes/workspace/rearc/rearc_landing/raw/bls/"
)

for file_name, table_name in metadata_files.items():

    print("Processing:", file_name)

    df = (
        spark.read
        .option("header", True)
        .option("sep", "\t")
        .option("inferSchema", False)
        .csv(base_path + file_name)
    )

    # Standardize column names
    cleaned_names = [
        re.sub(
            r"_+",
            "_",
            re.sub(r"[^a-zA-Z0-9_]", "_", c.strip().lower())
        ).strip("_")
        for c in df.columns
    ]

    assert len(cleaned_names) == len(set(cleaned_names)), \
        f"Duplicate columns in {file_name}"

    df = df.toDF(*cleaned_names)

    # Add lineage and audit columns
    df = (
        df
        .withColumn("_source_file", col("_metadata.file_path"))
        .withColumn("_ingestion_timestamp", current_timestamp())
    )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"workspace.rearc.{table_name}")
    )

    print("Created:", table_name, "| Rows:", df.count())

print("\nALL BLS REFERENCE BRONZE TABLES CREATED")
